# RAG : Usando  Qdrant como base de conocimiento

In [4]:
import os, getpass

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter

loader = WebBaseLoader(
    [
    "https://www.lavanguardia.com/andro4all/google/google-lanza-gemini-su-modelo-de-ia-mas-avanzado-hasta-la-fecha", 
    "https://www.xataka.com/robotica-e-ia/google-gemini-esta-aqui-asi-modelo-ia-avanzado-fecha-que-promete-ser-mejor-que-gpt-4",
    "https://www.infobae.com/tecno/2023/12/06/gemini-el-mejor-modelo-de-inteligencia-artificial-de-google-conoce-todos-los-detalles"
    ]
)

data = loader.load()

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 400, 
    chunk_overlap = 20
)

docs = text_splitter.split_documents(data)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
len(docs), docs

(19,
 [Document(metadata={'source': 'https://www.lavanguardia.com/andro4all/google/google-lanza-gemini-su-modelo-de-ia-mas-avanzado-hasta-la-fecha', 'title': 'Service Unavailable', 'language': 'No language found.'}, page_content='Service Unavailable\n\nService Unavailable - DNS failure\nThe server is temporarily unable to service your request.  Please try again\nlater.\nReference #11.e20c0317.1773892311.134715c0\nhttps://errors.edgesuite.net/11.e20c0317.1773892311.134715c0'),
  Document(metadata={'source': 'https://www.xataka.com/robotica-e-ia/google-gemini-esta-aqui-asi-modelo-ia-avanzado-fecha-que-promete-ser-mejor-que-gpt-4', 'title': 'Google Gemini ya está aquí: así es el modelo de IA más avanzado hasta la fecha que promete ser mejor que GPT-4', 'description': 'Google por fin muestra su gran arma contra ChatGPT. Gemini ya es oficial. El nuevo modelo de lenguaje (LLM) de Google ya se puede probar y promete ser la IA...', 'language': 'es'}, page_content='Google Gemini ya está aquí: a

## Cargando datos a Qrant

In [7]:
url = "https://00242789-d328-4c0f-8ab1-41aa7b2f5a29.us-east-1-1.aws.cloud.qdrant.io" #Reemplaza por tu url
api_key = getpass.getpass("Ingresa tu API Key de Qdrant : ")

In [8]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu API Key de OpenAI : ")

## Almacenando embeddings en Qdrant

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

embedding = OpenAIEmbeddings()

vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embedding,
    url = url,
    api_key = api_key,
    collection_name = "c_website",
    force_recreate = True
)

## Recuperando contexto de la colección en Qdrant

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

template = """Responde a la pregunta basándote solo en el siguiente contexto.
Si no puedes responder a la pregunta, responde exactamente:
"No lo sé disculpa, puedes buscar la información en internet."

<context>
{context}
</context>

Pregunta: {input}
Respuesta:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "input"]
)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0
)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)

qa = create_retrieval_chain(
    vectorstore.as_retriever(),
    combine_docs_chain
)


In [13]:
qa.invoke({"input": "¿qué es el modelo Gemini?"})["answer"]

'Gemini es un modelo de IA multimodal desarrollado por Google que supera a sus rivales en los principales tests, incluyendo a GPT-4 de OpenAI.'

In [14]:
qa.invoke({"input": "¿el modelo gemini es mejor que gpt4?"})["answer"]

'Sí, según Google, el modelo Gemini Ultra promete superar al resto de modelos de lenguaje, incluido GPT-4.'